In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("../src")
import pandas as pd
from difflib import SequenceMatcher
from areas import normalise, valid, build_gazetteer, extract_area

df = pd.read_parquet("../data/processed/karachi_events_clean.parquet")
gold = pd.read_csv("../data/processed/area_gold.csv").rename(columns={"eventid": "eventid_bad"})

# Label fixes, made before looking at any predictions
FIXES = {
    0: "Orangi", 1: "Gadap",                      # the two labels were swapped
    15: "Surjani|Rozi Goth",
    24: "Gadap|Gulshan-e-Maymar",
    28: "Sachal|Bilal Shah Norani",
    32: "Orangi",
    35: "Pirabad",
    39: "Liaquatabad|FC Area",
    44: "none",                                   # "Mubeena, India" is a data error
    45: "none",                                   # Malir was a former post, not the location
    47: "Gulshan-e-Iqbal|Gulshan",
}
for i, value in FIXES.items():
    gold.loc[i, "gold_area"] = value

# Excel damaged the eventid, so match events by their text instead
clean_text = lambda s: " ".join(str(s).split()) if pd.notna(s) else ""
for d in (gold, df):
    d["key"] = d["location"].map(clean_text) + "||" + d["summary_clean"].map(clean_text)
gold = gold.merge(df[["key", "eventid"]].drop_duplicates("key"), on="key", how="left")
print("Matched to events:", gold["eventid"].notna().sum(), "of", len(gold))

Matched to events: 60 of 60


In [2]:
from area_eval import add_keys, predictions, evaluate

df = add_keys(df)
gold = add_keys(gold.drop(columns=["key"]))
gold = gold.drop(columns=["eventid"], errors="ignore").merge(
    df[["key", "eventid"]].drop_duplicates("key"), on="key", how="left")

preds, known = predictions(df, "../data/processed/area_gliner_raw.parquet")
results = {n: evaluate(gold, p, n, known) for n, p in preds.items()}
print(pd.DataFrame([s for _, s in results.values()]).to_string(index=False))

method  coverage  precision  recall  recall_recurring  accuracy
 rules      0.68       1.00    0.85              0.98      0.88
gliner      0.62       0.95    0.73              0.83      0.77
hybrid      0.70       0.98    0.85              0.98      0.87


In [3]:
pd.set_option("display.max_colwidth", 60)
for name, (r, _) in results.items():
    bad = r[(r.has_gold & ~r.correct) | (~r.has_gold & r.predicted)]
    print(f"\n{name}: mistakes")
    print(bad[["gold", "pred", "location"]].to_string())


rules: mistakes
                               gold pred                                                     location
16                          FB Area  NaN         The incident took place in the al-Noor Society area.
27                     Zahra Garden  NaN  The incident occurred in the Zahra Garden area of the city.
41           Karachi Naval Dockyard  NaN                                                          NaN
42                       Memon Goth  NaN                                                          NaN
48  Pakistan State Oil Headquarters  NaN                                                          NaN
53                     Model Colony  NaN                                                          NaN
57                     Tawar Office  NaN                                                          NaN

gliner: mistakes
                               gold   pred                                                                            location
2                   Gu